# Day 20: Data Types and Type Conversion

**Dataset:** `Titanic_Dataset.csv`

This notebook inspects DataFrame data types and converts columns between strings, integers,
floats, booleans, and dates, using real values pulled directly from the Titanic dataset.

## 1. Import Libraries and Load the Dataset

In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv("../datasets/Titanic_Dataset.csv")

# Preview the data
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2. Data-Type Inspection Report

**`dtypes` — the data type of every column**

In [2]:
df.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

**`info()` — dtypes combined with non-null counts and memory usage**

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


**A closer look at which columns are numeric vs. text, and how many unique values each has**

In [4]:
dtype_report = pd.DataFrame({
    "dtype": df.dtypes,
    "non_null_count": df.count(),
    "unique_values": df.nunique()
})

dtype_report

,dtype,non_null_count,unique_values
PassengerId,int64,891,891
Survived,int64,891,2
Pclass,int64,891,3
Name,str,891,891
Sex,str,891,2
Age,float64,714,88
SibSp,int64,891,7
Parch,int64,891,7
Ticket,str,891,681
Fare,float64,891,248


**Interpretation:** `Survived` and `Pclass` are stored as plain integers, but both are
really **categories** (survived/died, and 1st/2nd/3rd class) — not quantities you'd add or
average meaningfully. `Sex` and `Embarked` are text with only 2–3 unique values each, which is
a good candidate for a memory-efficient `category` dtype rather than a general-purpose
string.

## 3. Converting `Survived`: integer (0/1) → boolean

In [5]:
print("Before:", df["Survived"].dtype)

df["Survived_bool"] = df["Survived"].astype(bool)

print("After:", df["Survived_bool"].dtype)
df[["Survived", "Survived_bool"]].head()

Before: int64
After: bool


,Survived,Survived_bool
0,0,False
1,1,True
2,1,True
3,1,True
4,0,False


## 4. Converting `PassengerId`: integer → string

`PassengerId` is an identifier, not a quantity — it's never meaningful to add two passenger
IDs together, so storing it as text better reflects what it actually represents.

In [6]:
print("Before:", df["PassengerId"].dtype)

df["PassengerId_str"] = df["PassengerId"].astype(str)

print("After:", df["PassengerId_str"].dtype)
df[["PassengerId", "PassengerId_str"]].head()

Before: int64
After: str


,PassengerId,PassengerId_str
0,1,1
1,2,2
2,3,3
3,4,4
4,5,5


## 5. Converting `Fare`: float → formatted string

In [7]:
print("Before:", df["Fare"].dtype)

df["Fare_formatted"] = df["Fare"].apply(lambda x: f"${x:,.2f}")

print("After:", df["Fare_formatted"].dtype)
df[["Fare", "Fare_formatted"]].head()

Before: float64
After: str


,Fare,Fare_formatted
0,7.2500,$7.25
1,71.2833,$71.28
2,7.9250,$7.92
3,53.1000,$53.10
4,8.0500,$8.05


**Using `pd.to_numeric()` to convert a string column back to numeric**

This is exactly the kind of situation `pd.to_numeric()` is built for — turning a text
representation of a number back into an actual numeric dtype, safely.

In [8]:
fare_back_to_numeric = pd.to_numeric(df["Fare_formatted"].str.replace("$", "").str.replace(",", ""))
print(fare_back_to_numeric.dtype)
print(fare_back_to_numeric.head())

# errors="coerce" turns anything that can't be converted into NaN instead of raising an error
sample_with_bad_value = pd.Series(["12.5", "30", "not_a_number"])
print(pd.to_numeric(sample_with_bad_value, errors="coerce"))

float64
0     7.25
1    71.28
2     7.92
3    53.10
4     8.05
Name: Fare_formatted, dtype: float64
0    12.5
1    30.0
2     NaN
dtype: float64


## 6. Converting `Embarked`: port code → real historical embarkation date

The Titanic's real 1912 voyage departed Southampton (`S`) and called at Cherbourg (`C`), both
on **April 10, 1912**, then Queenstown (`Q`) on **April 11, 1912**. Mapping the `Embarked` port
code to its real historical date gives a genuine string → datetime conversion example.

In [9]:
embarkation_dates = {
    "S": "1912-04-10",
    "C": "1912-04-10",
    "Q": "1912-04-11"
}

df["Embarkation_Date"] = pd.to_datetime(df["Embarked"].map(embarkation_dates))

print(df["Embarkation_Date"].dtype)
df[["Embarked", "Embarkation_Date"]].head()

datetime64[us]


,Embarked,Embarkation_Date
0,S,1912-04-10
1,C,1912-04-10
2,S,1912-04-10
3,S,1912-04-10
4,S,1912-04-10


## 7. A Cleaned Dataset with Appropriate Data Types

Combining the conversions above (and a couple more using `astype("category")`) into one final
DataFrame where each column's dtype matches what it actually represents.

In [10]:
df_cleaned = df.copy()

df_cleaned["Survived"] = df_cleaned["Survived"].astype(bool)
df_cleaned["Pclass"] = df_cleaned["Pclass"].astype("category")
df_cleaned["Sex"] = df_cleaned["Sex"].astype("category")
df_cleaned["Embarked"] = df_cleaned["Embarked"].astype("category")
df_cleaned["PassengerId"] = df_cleaned["PassengerId"].astype(str)
df_cleaned["Embarkation_Date"] = pd.to_datetime(df_cleaned["Embarked"].map(embarkation_dates))

# Drop the extra demo columns created earlier, since they're now redundant
df_cleaned = df_cleaned.drop(columns=["Survived_bool", "PassengerId_str", "Fare_formatted"], errors="ignore")

df_cleaned.dtypes

PassengerId                    str
Survived                      bool
Pclass                    category
Name                           str
Sex                       category
Age                        float64
SibSp                        int64
Parch                        int64
Ticket                         str
Fare                       float64
Cabin                          str
Embarked                  category
Embarkation_Date    datetime64[us]
dtype: object

In [11]:
df_cleaned.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Embarkation_Date
0,1,False,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,1912-04-10
1,2,True,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,1912-04-10
2,3,True,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1912-04-10
3,4,True,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,1912-04-10
4,5,False,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,1912-04-10


## Outcome

This notebook practiced inspecting Pandas data types with `dtypes` and `info()`, and converting
columns between different types using real values from the `Titanic_Dataset.csv` file. I
converted `Survived` from integer to boolean, `PassengerId` from integer to string (since it's
an identifier, not a quantity), `Fare` from float to a formatted string and back again using
`pd.to_numeric()` (including how `errors="coerce"` handles invalid values), and `Embarked`'s
port codes into real historical embarkation dates using `pd.to_datetime()`. Combining these
conversions, along with converting low-cardinality text columns like `Sex` and `Pclass` to the
memory-efficient `category` dtype, produced a fully cleaned dataset where every column's dtype
actually matches what that column represents, which matters because incorrect dtypes (like
treating a category as a plain number) can silently lead to meaningless calculations or wasted
memory.